In [3]:
import pandas as pd

# 1. Tabel Item Pesanan (Harga dan ongkir per item)
df_items = pd.read_csv('olist_order_items_dataset.csv')

# 2. Tabel Produk (Kategori dan dimensi produk)
df_products = pd.read_csv('olist_products_dataset.csv')

# 3. Tabel Terjemahan Kategori (Mengubah bahasa Portugis ke Inggris)
df_translation = pd.read_csv('product_category_name_translation.csv')

# 4. Tabel Pembayaran (Total nilai transaksi)
df_payments = pd.read_csv('olist_order_payments_dataset.csv')

# 5. Tabel Ulasan (Skor rating dari pelanggan)
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')

print("Semua dataset berhasil dimuat!")

Semua dataset berhasil dimuat!


In [4]:
# JOIN 1: Gabungkan df_items dengan df_products menggunakan 'product_id'
sales_data = pd.merge(df_items, df_products, on='product_id', how='left')

# JOIN 2: Terjemahkan nama kategori ke bahasa Inggris menggunakan 'product_category_name'
sales_data = pd.merge(sales_data, df_translation, on='product_category_name', how='left')

# JOIN 3: Tambahkan data pembayaran menggunakan 'order_id'
sales_data = pd.merge(sales_data, df_payments, on='order_id', how='left')

# JOIN 4: Tambahkan data ulasan pelanggan menggunakan 'order_id'
sales_data = pd.merge(sales_data, df_reviews, on='order_id', how='left')

# Hapus kolom kategori bahasa Portugis agar lebih rapi, sisakan versi bahasa Inggris
sales_data.drop(columns=['product_category_name'], inplace=True)
sales_data.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

# Tampilkan informasi dataset yang sudah digabung
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118310 entries, 0 to 118309
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   order_id                    118310 non-null  object 
 1   order_item_id               118310 non-null  int64  
 2   product_id                  118310 non-null  object 
 3   seller_id                   118310 non-null  object 
 4   shipping_limit_date         118310 non-null  object 
 5   price                       118310 non-null  float64
 6   freight_value               118310 non-null  float64
 7   product_name_lenght         116601 non-null  float64
 8   product_description_lenght  116601 non-null  float64
 9   product_photos_qty          116601 non-null  float64
 10  product_weight_g            118290 non-null  float64
 11  product_length_cm           118290 non-null  float64
 12  product_height_cm           118290 non-null  float64
 13  product_width_

In [11]:
sales_data.isna().sum()

order_id                           0
order_item_id                      0
product_id                         0
seller_id                          0
shipping_limit_date                0
price                              0
freight_value                      0
product_name_lenght             1695
product_description_lenght      1695
product_photos_qty              1695
product_weight_g                  20
product_length_cm                 20
product_height_cm                 20
product_width_cm                  20
product_category                   0
payment_sequential                 3
payment_type                       3
payment_installments               3
payment_value                      3
review_id                          0
review_score                       0
review_comment_title          103440
review_comment_message         67650
review_creation_date               0
review_answer_timestamp            0
dtype: int64

In [9]:
# 1. Mengecek dan Menghapus Data Duplikat
print(f"Jumlah baris sebelum hapus duplikat: {len(sales_data)}")
sales_data = sales_data.drop_duplicates()
print(f"Jumlah baris setelah hapus duplikat: {len(sales_data)}")

# 2. Mengatasi Kategori Produk yang Kosong
# Daripada menghapus datanya (yang membuat kita kehilangan data pendapatan), 
# kita ubah kategori yang kosong menjadi 'Unknown'
sales_data['product_category'] = sales_data['product_category'].fillna('Unknown')

# 3. Menangani Skor Ulasan yang Kosong
# Karena fokus kita mengukur kualitas produk, kita wajib memiliki angka pada 'review_score'.
# Kita hapus (drop) baris transaksi yang pelanggannya tidak memberikan rating.
sales_data = sales_data.dropna(subset=['review_score'])

print("\nPembersihan selesai! Data sudah siap dianalisis.")
print(f"Total baris akhir: {len(sales_data)}")

Jumlah baris sebelum hapus duplikat: 118310
Jumlah baris setelah hapus duplikat: 118310

Pembersihan selesai! Data sudah siap dianalisis.
Total baris akhir: 117332


In [12]:
# 1. Mengelompokkan data berdasarkan kategori dan menjumlahkan harganya
top_revenue = sales_data.groupby('product_category')['price'].sum().reset_index()

# 2. Mengurutkan dari pendapatan terbesar ke terkecil
top_revenue = top_revenue.sort_values(by='price', ascending=False)

# 3. Mengganti nama kolom agar lebih representatif
top_revenue.rename(columns={'price': 'total_revenue'}, inplace=True)

# 4. Menampilkan 10 kategori dengan pendapatan tertinggi
display(top_revenue.head(10))

,product_category,total_revenue
44,health_beauty,1291018.49
71,watches_gifts,1245783.11
8,bed_bath_table,1095770.05
66,sports_leisure,1022489.29
16,computers_accessories,944992.54
40,furniture_decor,765336.72
50,housewares,664226.76
21,cool_stuff,655820.40
6,auto,608914.30
43,garden_tools,515987.77


In [22]:
# 1. Menghitung rata-rata skor dan total ulasan per kategori
product_quality = sales_data.groupby('product_category').agg(
    avg_review_score=('review_score', 'mean'),
    total_reviews=('review_score', 'count')
).reset_index()

# 2. Memfilter hanya kategori yang memiliki lebih dari 50 ulasan (Statistical Significance)
product_quality = product_quality[product_quality['total_reviews'] > 50]

# 3. Mengurutkan dari rating terburuk (terendah) ke terbaik
worst_products = product_quality.sort_values(by='avg_review_score', ascending=True)

# 4. Menampilkan 10 kategori dengan ulasan terburuk
display(worst_products.head(10))

,product_category,avg_review_score,total_reviews
58,office_furniture,3.526791,1773
31,fashion_male_clothing,3.548611,144
35,fixed_telephony,3.672862,269
0,Unknown,3.833721,1720
5,audio,3.840849,377
39,furniture_bedroom,3.841270,126
20,construction_tools_safety,3.848485,198
8,bed_bath_table,3.890605,11847
48,home_confort,3.907563,476
40,furniture_decor,3.912158,8743


In [ ]:
# 1. Muat tabel orders (hanya ambil ID Pesanan dan Waktu Pembelian agar memori ringan)
df_orders = pd.read_csv('olist_orders_dataset.csv')[['order_id', 'order_purchase_timestamp']]

# 2. Gabungkan dengan data penjualan kita sebelumnya
sales_data = pd.merge(sales_data, df_orders, on='order_id', how='left')

# 3. Langkah Wajib: Ubah format teks (string) menjadi tipe data waktu (datetime)
sales_data['order_purchase_timestamp'] = pd.to_datetime(sales_data['order_purchase_timestamp'])

# 4. Ekspor
sales_data.to_csv('cleaned_olist_sales_data_v2.csv', index=False)
print("Data waktu berhasil ditambahkan! File v2 siap digunakan.")

Data berhasil diekspor dan siap untuk Tableau!
